# バイオ技術 1-2：RNA-seq解析の基礎

このNotebookでは、RNA-seq由来の発現量データを使って、**条件間で発現が変化している遺伝子を探す流れ**を体験します。

1-1では、Notebookの使い方、表データの読み込み、グラフ作成を学びました。  
ここからは少し専門的に、RNA-seq解析らしい内容に入ります。

## 今日のゴール

このNotebookが終わるころには、以下のことを説明できるようになることを目指します。

- RNA-seqデータが「遺伝子 × サンプル」の表であることがわかる
- サンプル同士が似ているかを確認できる
- PCAやクラスタリングでサンプルの違いを可視化できる
- log2 fold change と p値の意味を大まかに説明できる
- ボルケーノプロットを読める
- 発現変動遺伝子（DEG）の候補リストを作れる
- DEGヒートマップから、条件ごとの発現パターンを読み取れる

## 大事なメッセージ

RNA-seq解析は、図を描いて終わりではありません。  
**「どの遺伝子が変化していそうか」を見つけ、その後のwet実験の候補を考えるための解析**です。

このNotebookでは、教育用にPythonだけで簡単な解析を行います。  
実際の研究では、DESeq2、edgeR、limma-voomなど、RNA-seq専用の手法を使うことが多いです。


## このNotebookの進め方

このNotebookでは、セルを順番に実行するだけでなく、途中に4つのミニ演習があります。

- 比較するサンプルを自分で選ぶ

- PCAを自分の言葉で説明する

- DEGの閾値を自分で決める

- 候補遺伝子を自分で選ぶ

「実行する → 結果を見る → 自分で考える」を繰り返しながら進めてください。


---
## 0. 今回の研究の問い

今回は、2つの条件のRNA-seqデータを比較します。

- `batch`：batch培養のサンプル
- `chemostat`：chemostat培養のサンプル

それぞれ3サンプルずつあります。

このNotebookで考える問いは、次の1つです。

> **batch培養とchemostat培養で、発現量が大きく変化している遺伝子はどれか？**

解析の流れは次のようになります。

1. データを読み込む
2. サンプル情報を確認する
3. サンプル同士が似ているか確認する
4. 条件間で発現量を比較する
5. ボルケーノプロットを作る
6. DEGを抽出する
7. DEGの発現パターンをヒートマップで見る


---
## 1. ライブラリを読み込む

まず、解析に使うライブラリを読み込みます。

| ライブラリ | 役割 |
|---|---|
| pandas | 表データを扱う |
| numpy | 数値計算を行う |
| matplotlib / seaborn | グラフを描く |
| scipy | 統計解析やクラスタリングを行う |
| scikit-learn | PCAを行う |

ここでは中身をすべて覚える必要はありません。  
「解析に使う道具を準備している」と考えてください。


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import ttest_ind
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.decomposition import PCA

# 図のサイズや見た目を少し整える
sns.set_context("notebook")
plt.rcParams["figure.figsize"] = (8, 5)


---
## 2. TPMデータを読み込む

RNA-seq解析では、まず発現量データを読み込みます。

今回使うデータは、TPMという単位で表された発現量データです。

TPMは、簡単に言うと、

> それぞれの遺伝子が、各サンプルでどれくらい発現しているか

を表した値です。

下のセルを実行して、Excelファイルを読み込みます。


In [ ]:
DATA_URL = "https://github.com/iwata97/bioinfo/raw/refs/heads/main/TPM_data.xlsx"

# Excelファイルを読み込む
df_tpm = pd.read_excel(DATA_URL)

# 最初の5行を表示する
df_tpm.head()


### 確認ポイント

表示された表を見て、次のことを確認してください。

- 1行目には列名がある
- `gene`列には遺伝子名が入っている
- それ以外の列には各サンプルのTPM値が入っている
- 行は遺伝子、列はサンプルを表している


---
## 3. データの大きさを確認する

データ解析では、最初に「どれくらいの大きさのデータなのか」を確認することが大切です。

ここでは、行数と列数を確認します。


In [ ]:
print("データの行数・列数:")
print(df_tpm.shape)

print("\n列名:")
print(df_tpm.columns.tolist())


### 読み方

`df_tpm.shape` の結果は、

```text
(行数, 列数)
```

の順に表示されます。

このデータでは、行数がおおよそ遺伝子数、列数が `gene`列 + サンプル数 を表しています。


---
## 4. サンプル情報を整理する

次に、どの列がどの条件のサンプルなのかを整理します。

RNA-seq解析では、発現量データだけでなく、

- どのサンプルがどの条件か
- どのサンプル同士を比較するか

を正しく把握することが重要です。


In [ ]:
# gene列以外をサンプル列として取り出す
sample_cols = [col for col in df_tpm.columns if col != "gene"]

# サンプル名から条件名を推定する
sample_info = pd.DataFrame({"sample": sample_cols})
sample_info["group"] = sample_info["sample"].apply(
    lambda x: "batch" if "batch" in x.lower() else "chemostat" if "chemo" in x.lower() else "unknown"
)

sample_info


### 確認ポイント

ここで、サンプルが次のように分類されていることを確認してください。

- `batch`：batch培養のサンプル
- `chemostat`：chemostat培養のサンプル

もし `unknown` が表示された場合は、サンプル名から条件名をうまく読み取れていない可能性があります。


In [ ]:
batch_samples = sample_info.loc[sample_info["group"] == "batch", "sample"].tolist()
chemo_samples = sample_info.loc[sample_info["group"] == "chemostat", "sample"].tolist()

print("batch samples:", batch_samples)
print("chemostat samples:", chemo_samples)


---
## 5. 解析用の発現量テーブルを作る

元の表には `gene` 列があります。  
計算をしやすくするために、遺伝子名を行名にして、発現量だけの表を作ります。

また、TPM値は非常に大きい値と小さい値が混ざっています。  
そのままだと見にくいため、`log2(TPM + 1)` に変換します。

`+1` するのは、TPMが0の遺伝子でもlog変換できるようにするためです。


In [ ]:
# gene列を行名にして、サンプル列だけを取り出す
tpm = df_tpm.set_index("gene")[sample_cols]

# 数値として扱えない値があれば欠損値に変換する
tpm = tpm.apply(pd.to_numeric, errors="coerce")

# 同じ遺伝子名が複数行ある場合は、平均値で1行にまとめる
# （教育用データで重複遺伝子名があっても、後の解析が安定して動くようにする）
tpm = tpm.groupby(level=0).mean()

# log2(TPM + 1) に変換する
log_tpm = np.log2(tpm + 1)

print("TPM table:")
display(tpm.head())

print("\nlog2(TPM + 1) table:")
display(log_tpm.head())

print("\n解析対象の遺伝子数:", len(log_tpm))


### 考えてみよう

元のTPM値と、log変換後の値を比べてみましょう。

- 数値の範囲はどう変わりましたか？
- log変換すると、極端に大きい値の影響はどうなりそうですか？


---
## 6. サンプルごとの発現量分布を見る

まず、各サンプルの発現量分布を見ます。

ここでは、log変換後の発現量を箱ひげ図で表示します。  
サンプル間で全体的な分布が大きく違いすぎる場合、解析前に注意が必要です。


In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=log_tpm)
plt.xticks(rotation=45)
plt.ylabel("log2(TPM + 1)")
plt.title("Expression distribution of each sample")
plt.show()


### 見るポイント

- すべてのサンプルで、発現量分布は大きく似ていますか？
- 1つだけ極端に違うサンプルはありませんか？

RNA-seq解析では、いきなりDEGを探す前に、まずサンプル全体の様子を確認します。


---
## 7. サンプル間の相関を見る

次に、サンプル同士がどれくらい似ているかを見ます。

同じ条件のサンプル同士は、似た発現パターンを示すことが期待されます。  
そのため、サンプル間相関は、データの再現性を確認するための基本的な可視化です。


In [ ]:
# サンプル間の相関係数を計算する
sample_corr = log_tpm.corr()

plt.figure(figsize=(7, 6))
sns.heatmap(sample_corr, annot=True, cmap="vlag", vmin=0, vmax=1)
plt.title("Sample correlation heatmap")
plt.show()


### ミニ演習1：自分で2つのサンプルを選んで相関を見る

ヒートマップを眺めるだけでなく、2つのサンプルを自分で選んで相関係数を計算してみましょう。

まずは、次の2通りを試してください。

1. 同じ条件の2サンプル
2. 異なる条件の2サンプル

**どちらの相関が高くなると予想しますか？**  
先に予想してから、下のセルの数字を書き換えて実行してください。


In [ ]:
# ↓↓↓ 0〜5の数字を書き換えて、比較するサンプルを選んでください ↓↓↓
sample_a_index = 0
sample_b_index = 1

sample_a = sample_cols[sample_a_index]
sample_b = sample_cols[sample_b_index]

r = log_tpm[sample_a].corr(log_tpm[sample_b])

print("sample A:", sample_a)
print("sample B:", sample_b)
print(f"correlation r = {r:.3f}")


#### 記録してみよう

- 同じ条件の2サンプルの相関係数：
- 異なる条件の2サンプルの相関係数：
- 予想と同じでしたか？：


### 読み方

相関係数は、2つのサンプルがどれくらい似ているかを表す値です。

- 1に近い：よく似ている
- 0に近い：あまり似ていない

ここでは、同じ条件のサンプル同士が似ているか、異なる条件間では少し違いがあるかを見ます。


---
## 8. 階層的クラスタリングでサンプルを並べる

相関ヒートマップでは、サンプル同士の似ている度合いを数字で見ました。  
次に、階層的クラスタリングを使って、似ているサンプル同士が近くに並ぶかを確認します。

階層的クラスタリングでは、似ているもの同士を順番にまとめて、樹形図として表示します。


In [ ]:
# サンプルが行、遺伝子が列になるように転置する
log_tpm_t = log_tpm.T

# 階層的クラスタリング
linkage_result = linkage(log_tpm_t, method="ward", metric="euclidean")

plt.figure(figsize=(9, 5))
dendrogram(linkage_result, labels=list(log_tpm_t.index))
plt.ylabel("Distance")
plt.title("Hierarchical clustering of samples")
plt.show()


### 考えてみよう

- `batch` のサンプル同士は近くに並んでいますか？
- `chemostat` のサンプル同士は近くに並んでいますか？
- 条件の違いが、発現量全体の違いとして見えていそうですか？


---
## 9. PCAでサンプルの違いを見る

PCAは、多数の遺伝子発現情報を少数の軸にまとめて、サンプル間の違いを見やすくする方法です。

ここでは、各サンプルを2次元上の点として表示します。

- 近くにある点：発現パターンが似ている
- 離れている点：発現パターンが異なる

1-3のsingle-cell解析では、似た考え方でUMAPを使います。


In [ ]:
# PCAを実行する
pca = PCA(n_components=2)
pca_result = pca.fit_transform(log_tpm_t)

pca_df = pd.DataFrame(pca_result, columns=["PC1", "PC2"])
pca_df["sample"] = log_tpm_t.index
pca_df = pca_df.merge(sample_info, on="sample", how="left")

# PCAプロット
plt.figure(figsize=(7, 6))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="group", s=120)

for _, row in pca_df.iterrows():
    plt.text(row["PC1"], row["PC2"], row["sample"], fontsize=9, ha="left", va="bottom")

pc1_var = pca.explained_variance_ratio_[0] * 100
pc2_var = pca.explained_variance_ratio_[1] * 100
plt.xlabel(f"PC1 ({pc1_var:.1f}%)")
plt.ylabel(f"PC2 ({pc2_var:.1f}%)")
plt.title("PCA plot of samples")
plt.legend(title="group")
plt.show()


### ミニ演習2：PCAを自分の言葉で説明する

PCAプロットを見て、次の3点を短く書いてください。

1. batchとchemostatは分かれて見えますか？
2. 同じ条件の3サンプルは近くに集まっていますか？
3. この結果から、培養条件の違いは発現パターン全体に影響していそうですか？

**答え：**

-
-
-


### 見るポイント

PCAでは、点の位置関係を見ます。

- 同じ条件のサンプルは近くにありますか？
- batchとchemostatは分かれて見えますか？
- PC1とPC2のどちらが条件差をよく表していそうですか？


---
## 10. 条件ごとの平均発現量を比べる

ここから、条件間で発現が変化している遺伝子を探します。

まずは、各遺伝子について、

- batch群の平均発現量
- chemostat群の平均発現量

を計算し、散布図で比較します。

対角線から大きく離れている遺伝子は、2つの条件で発現量が違う可能性があります。


In [ ]:
batch_mean = log_tpm[batch_samples].mean(axis=1)
chemo_mean = log_tpm[chemo_samples].mean(axis=1)

plt.figure(figsize=(6, 6))
plt.scatter(batch_mean, chemo_mean, alpha=0.3, s=10)

# 対角線を描く
min_val = min(batch_mean.min(), chemo_mean.min())
max_val = max(batch_mean.max(), chemo_mean.max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")

plt.xlabel("Mean expression in batch")
plt.ylabel("Mean expression in chemostat")
plt.title("Mean expression comparison")
plt.show()


### 読み方

- 対角線上に近い点：batchとchemostatであまり変わらない遺伝子
- 対角線より上の点：chemostatで高い遺伝子
- 対角線より下の点：batchで高い遺伝子

ただし、この図だけでは、サンプル間のばらつきや統計的な有意性は考慮できません。  
そこで次に、log2 fold changeとp値を計算します。


---
## 11. log2 fold change と p値を計算する

条件間で発現が変化している遺伝子を探すために、2つの指標を計算します。

| 指標 | 意味 |
|---|---|
| log2 fold change | どれくらい発現量が変化したか |
| p値 | その差が偶然でも起こりそうか |

ここでは、教育用にWelchのt検定を使います。  
実際のRNA-seq研究では、カウントデータの性質を考慮した専用手法を使うことが多いです。

## log2 fold changeの向き

このNotebookでは、次のように計算します。

```text
log2FC = chemostat群の平均 - batch群の平均
```

したがって、

- log2FC > 0：chemostatで高い
- log2FC < 0：batchで高い

と読みます。


In [ ]:
def benjamini_hochberg(p_values):
    """Benjamini-Hochberg法でp値を補正する簡易関数"""
    p_values = np.asarray(p_values, dtype=float)
    n = len(p_values)
    order = np.argsort(p_values)
    ranked_p = p_values[order]
    adjusted = ranked_p * n / np.arange(1, n + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.clip(adjusted, 0, 1)
    q_values = np.empty(n)
    q_values[order] = adjusted
    return q_values

# log2 fold changeを計算
log2fc = chemo_mean - batch_mean

# 各遺伝子についてWelchのt検定を行う
# 行＝遺伝子、列＝反復サンプルとして、全遺伝子を一度に計算する
chemo_array = log_tpm[chemo_samples].to_numpy(dtype=float)
batch_array = log_tpm[batch_samples].to_numpy(dtype=float)

stat, pvals = ttest_ind(
    chemo_array,
    batch_array,
    axis=1,
    equal_var=False,
    nan_policy="omit"
)

# 計算できなかったp値（NaN）があれば1.0として扱う
pvals = np.asarray(pvals, dtype=float)
pvals = np.where(np.isfinite(pvals), pvals, 1.0)

qvals = benjamini_hochberg(pvals)
neg_log10_pvals = -np.log10(pvals + 1e-300)

# 結果を1つの表にまとめる
results_df = pd.DataFrame({
    "gene": log_tpm.index.to_numpy(),
    "batch_mean": batch_mean.to_numpy(),
    "chemostat_mean": chemo_mean.to_numpy(),
    "log2FC": log2fc.to_numpy(),
    "pval": pvals,
    "qval_BH": qvals,
    "minus_log10_pval": neg_log10_pvals
})

results_df.head()


### 補足：p値とq値

多数の遺伝子を同時に検定すると、偶然に有意に見える遺伝子も出てきます。  
そのため、実際の解析では多重検定補正を行い、補正後の値を見ます。

ここでは、Benjamini-Hochberg法で補正した値を `qval_BH` として計算しています。  
ただし、今回は教育用の簡易解析なので、まずはp値とlog2FCの読み方を重視します。


---
## 12. ボルケーノプロットを描く

ボルケーノプロットは、RNA-seq解析でよく使われる図です。

- 横軸：log2 fold change
- 縦軸：-log10(p値)

つまり、

- 右側：chemostatで高い遺伝子
- 左側：batchで高い遺伝子
- 上側：p値が小さい遺伝子

を表します。


In [ ]:
# DEGの閾値を設定する
LOG2FC_THRESHOLD = 2
PVAL_THRESHOLD = 0.05

results_df["is_DEG"] = (results_df["log2FC"].abs() > LOG2FC_THRESHOLD) & (results_df["pval"] < PVAL_THRESHOLD)
results_df["direction"] = "not_DEG"
results_df.loc[(results_df["is_DEG"]) & (results_df["log2FC"] > 0), "direction"] = "chemostat_high"
results_df.loc[(results_df["is_DEG"]) & (results_df["log2FC"] < 0), "direction"] = "batch_high"

plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=results_df,
    x="log2FC",
    y="minus_log10_pval",
    hue="direction",
    alpha=0.6,
    s=20
)

plt.axhline(-np.log10(PVAL_THRESHOLD), linestyle="--")
plt.axvline(LOG2FC_THRESHOLD, linestyle="--")
plt.axvline(-LOG2FC_THRESHOLD, linestyle="--")

plt.xlabel("log2 fold change (chemostat - batch)")
plt.ylabel("-log10(p-value)")
plt.title("Volcano plot")
plt.legend(title="gene group")
plt.show()


### ミニ演習3：自分でDEGの基準を決める

研究では、候補遺伝子を選ぶ基準を自分で決める必要があります。

下のセルで、

- `MY_LOG2FC_THRESHOLD`
- `MY_PVAL_THRESHOLD`

を自分で設定してください。

例えば、次のような考え方があります。

- 候補を広く拾いたい → 閾値を少しゆるくする
- wet実験で検証できる数に絞りたい → 閾値を厳しくする

正解は1つではありません。  
**設定した理由も考えてみてください。**


In [ ]:
# ↓↓↓ 自分で値を決めて書き換えてください ↓↓↓
MY_LOG2FC_THRESHOLD = 1.5
MY_PVAL_THRESHOLD = 0.05

my_deg = (
    (results_df["log2FC"].abs() > MY_LOG2FC_THRESHOLD)
    & (results_df["pval"] < MY_PVAL_THRESHOLD)
)

my_chemo_high = (
    (results_df["log2FC"] > MY_LOG2FC_THRESHOLD)
    & (results_df["pval"] < MY_PVAL_THRESHOLD)
)

my_batch_high = (
    (results_df["log2FC"] < -MY_LOG2FC_THRESHOLD)
    & (results_df["pval"] < MY_PVAL_THRESHOLD)
)

print("あなたの基準")
print("log2FC threshold:", MY_LOG2FC_THRESHOLD)
print("p-value threshold:", MY_PVAL_THRESHOLD)
print()
print("DEG total:", int(my_deg.sum()))
print("chemostat high:", int(my_chemo_high.sum()))
print("batch high:", int(my_batch_high.sum()))


#### 記録してみよう

- 設定したlog2FC閾値：
- 設定したp値閾値：
- 得られたDEG数：
- この基準を選んだ理由：


### 考えてみよう

- 右上にある遺伝子は、どちらの条件で高いですか？
- 左上にある遺伝子は、どちらの条件で高いですか？
- DEGとして選ばれるには、横軸と縦軸の両方でどのような条件を満たす必要がありますか？


---
## 13. DEGの数を数える

ボルケーノプロットで色がついた遺伝子が、今回の基準で選ばれたDEGです。  
次に、DEGが何個あるかを数えてみます。


In [ ]:
deg_counts = results_df["direction"].value_counts()
deg_counts


In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=results_df, x="direction", order=["batch_high", "chemostat_high", "not_DEG"])
plt.xlabel("Gene group")
plt.ylabel("Number of genes")
plt.title("Number of DEG and non-DEG genes")
plt.xticks(rotation=20)
plt.show()


### 読み方

- `chemostat_high`：chemostatで高いDEG
- `batch_high`：batchで高いDEG
- `not_DEG`：今回の基準ではDEGではない遺伝子

DEGの数は、使う閾値によって変わります。  
次に、閾値を変えると結果がどう変わるかを見ます。


---
## 14. 閾値を変えるとDEG数はどう変わるか

研究では、どの基準で候補遺伝子を選ぶかが重要です。

閾値をゆるくすると候補は増えますが、偽陽性も増える可能性があります。  
閾値を厳しくすると候補は減りますが、重要な遺伝子を見逃す可能性もあります。

ここでは、いくつかの閾値でDEG数を比較します。


In [ ]:
threshold_settings = [
    {"name": "loose", "log2FC": 1.0, "pval": 0.05},
    {"name": "standard", "log2FC": 2.0, "pval": 0.05},
    {"name": "strict", "log2FC": 2.0, "pval": 0.01},
    {"name": "very_strict", "log2FC": 3.0, "pval": 0.01},
]

rows = []
for setting in threshold_settings:
    is_deg = (results_df["log2FC"].abs() > setting["log2FC"]) & (results_df["pval"] < setting["pval"])
    up = ((results_df["log2FC"] > setting["log2FC"]) & (results_df["pval"] < setting["pval"])).sum()
    down = ((results_df["log2FC"] < -setting["log2FC"]) & (results_df["pval"] < setting["pval"])).sum()
    rows.append({
        "setting": setting["name"],
        "log2FC_threshold": setting["log2FC"],
        "pval_threshold": setting["pval"],
        "DEG_total": int(is_deg.sum()),
        "chemostat_high": int(up),
        "batch_high": int(down),
    })

threshold_df = pd.DataFrame(rows)
threshold_df


### 考えてみよう

- 閾値を厳しくすると、DEG数はどうなりましたか？
- 候補遺伝子が多すぎる場合、wet実験では何が困りそうですか？
- 候補遺伝子が少なすぎる場合、何が問題になりそうですか？


---
## 15. 上位DEGを確認する

次に、DEGとして選ばれた遺伝子の中から、p値が小さい順に上位を表示します。

ここでは、

- chemostatで高い遺伝子
- batchで高い遺伝子

を分けて確認します。


In [ ]:
chemostat_high_df = results_df[results_df["direction"] == "chemostat_high"].sort_values("pval")
batch_high_df = results_df[results_df["direction"] == "batch_high"].sort_values("pval")

print("chemostatで高いDEG 上位10個")
display(chemostat_high_df[["gene", "log2FC", "pval", "qval_BH", "batch_mean", "chemostat_mean"]].head(10))

print("batchで高いDEG 上位10個")
display(batch_high_df[["gene", "log2FC", "pval", "qval_BH", "batch_mean", "chemostat_mean"]].head(10))


### ミニ演習4：候補遺伝子を3つ予備選択する

上に表示された表を見て、気になる遺伝子を3つ選んでください。

この時点では、遺伝子機能を知らなくても構いません。  
次の情報を見て、自分なりの理由で選びます。

- log2FCが大きい
- p値が小さい
- 発現量そのものが高い
- batchとchemostatの差が明確
- 遺伝子名が気になる

#### 選んだ候補

1. 遺伝子名：  
   選んだ理由：

2. 遺伝子名：  
   選んだ理由：

3. 遺伝子名：  
   選んだ理由：


### 見るポイント

候補遺伝子を選ぶときは、1つの指標だけで決めるのではなく、複数の情報を見ます。

- log2FCは大きいか
- p値は小さいか
- 発現量そのものは十分にあるか
- 条件間で一貫して変わっていそうか
- 遺伝子の機能として説明できそうか

午後のミニ研究では、このような視点で候補遺伝子を選びます。


---
## 16. DEGヒートマップを描く

ボルケーノプロットでは、各遺伝子を点として見ました。  
次に、DEGの発現パターンをヒートマップで確認します。

ヒートマップでは、

- 行：遺伝子
- 列：サンプル
- 色：発現量の高低

を表します。

ここでは、DEGのうちp値が小さい上位30遺伝子を表示します。  
DEGが30個未満の場合は、表示できる範囲で表示します。


In [ ]:
# DEGがある場合はDEG上位30、ない場合はp値上位30を表示する
heatmap_genes_df = results_df[results_df["is_DEG"]].sort_values("pval").head(30)

if len(heatmap_genes_df) == 0:
    print("現在の閾値ではDEGがありません。p値が小さい上位30遺伝子を表示します。")
    heatmap_genes_df = results_df.sort_values("pval").head(30)

heatmap_genes = heatmap_genes_df["gene"].tolist()

# 遺伝子ごとに標準化して、サンプル間の相対的な発現差を見やすくする
heatmap_data = log_tpm.loc[heatmap_genes]
heatmap_data_z = heatmap_data.sub(heatmap_data.mean(axis=1), axis=0)
heatmap_data_z = heatmap_data_z.div(heatmap_data.std(axis=1).replace(0, np.nan), axis=0)
heatmap_data_z = heatmap_data_z.fillna(0)

sns.clustermap(
    heatmap_data_z,
    cmap="vlag",
    figsize=(9, 10),
    col_cluster=True,
    row_cluster=True,
    yticklabels=True
)
plt.suptitle("Heatmap of top DEG", y=1.02)
plt.show()


### 読み方

このヒートマップでは、各遺伝子について、サンプル間で相対的に高い・低いを見ています。

- 赤っぽい：その遺伝子の中では発現が高い
- 青っぽい：その遺伝子の中では発現が低い

見るポイントは次の通りです。

- batchサンプルとchemostatサンプルが分かれて見えるか
- DEGとして選ばれた遺伝子は、条件ごとに一貫した発現パターンを示しているか
- 1つのサンプルだけが極端な値になっていないか


---
## 17. 1つの候補遺伝子の発現量を詳しく見る

候補遺伝子を選ぶときには、ボルケーノプロットやヒートマップだけでなく、1つの遺伝子の発現量をサンプルごとに確認すると理解しやすくなります。

下のセルでは、1つの遺伝子を選んで、batch群とchemostat群で発現量を比較します。

まずは、DEG上位の遺伝子を自動的に1つ選びます。  
あとで `gene_name` を別の遺伝子名に書き換えて試してみましょう。


In [ ]:
# 上の「上位DEG」の表を見て、気になる遺伝子名を1つ入力してください
# まずは候補表の先頭遺伝子を初期値にしています
candidate_table = results_df[results_df["is_DEG"]].sort_values("pval")

if len(candidate_table) == 0:
    candidate_table = results_df.sort_values("pval")

print("候補遺伝子の例:")
display(candidate_table[["gene", "log2FC", "pval", "batch_mean", "chemostat_mean"]].head(10))

# ↓↓↓ ここを書き換えてください ↓↓↓
gene_name = candidate_table.iloc[0]["gene"]

print("表示する遺伝子:", gene_name)

if gene_name not in log_tpm.index:
    raise ValueError("入力した遺伝子名が見つかりません。上の表からコピーしてください。")

plot_df = pd.DataFrame({
    "sample": sample_cols,
    "expression": log_tpm.loc[gene_name, sample_cols].to_numpy()
})
plot_df = plot_df.merge(sample_info, on="sample", how="left")

plt.figure(figsize=(6, 5))
sns.stripplot(data=plot_df, x="group", y="expression", size=10)
sns.pointplot(
    data=plot_df,
    x="group",
    y="expression",
    errorbar="sd",
    join=False,
    markers="_",
    scale=1.5
)
plt.ylabel("log2(TPM + 1)")
plt.title(f"Expression of {gene_name}")
plt.show()

plot_df


### 小改造

上のセルの `gene_name` を、自分が気になる遺伝子名に書き換えてみましょう。

例：

```python
 gene_name = "ここに遺伝子名を入れる"
```

午後のミニ研究では、このように候補遺伝子を1つずつ確認していきます。


---
## 18. DEGリストを保存する

最後に、DEGリストをCSVファイルとして保存します。

CSVファイルはExcelでも開けます。  
午後のミニ研究では、このリストを使って候補遺伝子を選びます。


In [ ]:
# DEGだけを抽出して、p値順に並べる
deg_df = results_df[results_df["is_DEG"]].sort_values("pval")

# CSVファイルとして保存する
deg_df.to_csv("DEG_list.csv", index=False)
results_df.to_csv("RNAseq_all_gene_results.csv", index=False)

print("保存しました:")
print("- DEG_list.csv")
print("- RNAseq_all_gene_results.csv")

print("\nDEGリストの先頭:")
display(deg_df.head(10))


---
## 19. ミニ確認問題

Notebookの最後に、今日の内容を自分の言葉で整理してみましょう。

### Q1. RNA-seqデータの表では、行と列は何を表していましたか？

- 行：
- 列：

### Q2. PCAや階層的クラスタリングでは、何を確認しましたか？

答え：

### Q3. log2FC > 0 の遺伝子は、どちらの条件で発現が高いですか？

答え：

### Q4. ボルケーノプロットの右上にある遺伝子は、どのような遺伝子ですか？

答え：

### Q5. DEGリストは、wet研究ではどのように使えそうですか？

答え：


---
## 20. よくあるエラーと対処

### `NameError: name 'log_tpm' is not defined`

途中のセルを飛ばして実行している可能性があります。  
Notebookは基本的に上から順番に実行してください。

### `KeyError: 'gene'`

`gene`列が見つからないというエラーです。  
データが正しく読み込めているか、列名が変わっていないか確認してください。

### グラフが表示されない

セルがまだ実行されていない可能性があります。  
左側の ▶ ボタン、または `Shift + Enter` で実行してください。

### DEGが少ない、または出ない

閾値が厳しすぎる可能性があります。  
`LOG2FC_THRESHOLD` や `PVAL_THRESHOLD` を変えると、DEG数が変わります。

---
## 次へ

このNotebookでは、RNA-seq解析の基本的な流れを体験しました。

次の1-3では、single-cell RNA-seq解析を扱います。  
1-2では「サンプル」を比較しましたが、1-3では「細胞1個1個」を比較します。

午後の発展演習では、1-2で作ったDEGリストを使って、候補遺伝子を選び、次に行うwet実験を考えます。
